In [0]:
import numpy as np
import pandas as pd

df = (spark.table("workspace.default.partd_nc_2019")
      .unionByName(spark.table("workspace.default.partd_nc_2020"), allowMissingColumns=True)
      .unionByName(spark.table("workspace.default.partd_nc_2021"), allowMissingColumns=True)
      .toPandas())
print(f"Start: {len(df):,} rows, {df['Prscrbr_NPI'].nunique():,} prescribers")

In [0]:
print(df["Prscrbr_Ent_Cd"].value_counts(dropna=False))
df = df[df["Prscrbr_Ent_Cd"] == "I"].copy()
ind = df.copy()   # snapshot, used later to see who the filters dropped
print(f"After individuals only: {len(df):,} rows")

In [0]:
n0 = len(df)
df = df[df["Tot_Clms"] >= 50].copy()
print(f"After 50-claim cutoff: {len(df):,} rows (dropped {n0 - len(df):,})")

In [0]:
years = df.groupby("Prscrbr_NPI")["year"].nunique()
keep = years[years >= 2].index
df = df[df["Prscrbr_NPI"].isin(keep)].copy()
print(f"After 2+ years: {len(df):,} rows, {df['Prscrbr_NPI'].nunique():,} prescribers")

In [0]:
types = ["Nurse Practitioner", "Physician Assistant", "Family Practice", "Internal Medicine", "Dentist"]
before = ind[ind["Prscrbr_Type"].isin(types)].groupby("Prscrbr_Type")["Prscrbr_NPI"].nunique()
after = df[df["Prscrbr_Type"].isin(types)].groupby("Prscrbr_Type")["Prscrbr_NPI"].nunique()
kept = pd.DataFrame({"before": before, "after": after})
kept["kept_%"] = (kept["after"] / kept["before"] * 100).round(1)
kept.sort_values("kept_%")

In [0]:
df.attrs = {}   
df.to_parquet("/Volumes/workspace/default/raw/partd_nc_clean.parquet", index=False)
print("Saved", len(df), "rows")

####Load the clean data and check what's hidden

In [0]:
d = pd.read_parquet("/Volumes/workspace/default/raw/partd_nc_clean.parquet")

# 1. What do the privacy flags say?
for c in ["Brnd_Sprsn_Flag", "Gnrc_Sprsn_Flag", "Othr_Sprsn_Flag", "GE65_Sprsn_Flag", "Antpsyct_GE65_Sprsn_Flag"]:
    print(c, d[c].value_counts(dropna=False).to_dict())

# 2. How much is blank in the columns our features need?
cols = ["Brnd_Tot_Clms", "Gnrc_Tot_Clms", "Othr_Tot_Clms", "Opioid_Tot_Clms", "Opioid_Tot_Benes",
        "Opioid_Tot_Suply", "Opioid_LA_Tot_Clms", "Antbtc_Tot_Clms", "GE65_Tot_Clms",
        "Antpsyct_GE65_Tot_Clms", "Tot_Benes", "LIS_Tot_Clms"]
(d[cols].isna().mean() * 100).round(1).sort_values(ascending=False)

##### Check the math first

In [0]:
# Does brand + generic + other really equal total? (rows where all three are shown)
parts = ["Brnd_Tot_Clms", "Gnrc_Tot_Clms", "Othr_Tot_Clms"]
shown = d[parts].notna().all(axis=1)
diff = (d.loc[shown, "Tot_Clms"] - d.loc[shown, parts].sum(axis=1)).abs()
print(f"All three shown: {shown.sum():,} rows | adds up exactly: {(diff < 0.5).mean():.1%}")

# Which combinations of brand/other flags happen?
print(pd.crosstab(d["Brnd_Sprsn_Flag"].fillna("shown"), d["Othr_Sprsn_Flag"].fillna("shown")))

# When someone has 0 opioids, are the other opioid columns 0 or blank?
z = d["Opioid_Tot_Clms"] == 0
print("0 opioids:", z.sum(), "| long-acting blank:", d.loc[z, "Opioid_LA_Tot_Clms"].isna().sum(),
      "| opioid patients blank:", d.loc[z, "Opioid_Tot_Benes"].isna().sum())

#### Fill

In [0]:
f = d.copy()

# --- Brand: back-calculate when hidden ---
rest = f["Tot_Clms"] - f["Gnrc_Tot_Clms"]                      # brand + other
other_shown = f["Othr_Sprsn_Flag"].isna()
other_est = f["Othr_Tot_Clms"].where(other_shown, 5)
brand = f["Brnd_Tot_Clms"].copy()
brand[f["Brnd_Sprsn_Flag"].eq("*")] = 5
is_hash = f["Brnd_Sprsn_Flag"].eq("#")
brand[is_hash] = (rest - other_est)[is_hash]
exact = f["Brnd_Sprsn_Flag"].notna() & other_shown              # other known, so brand is exact
brand[exact] = (rest - f["Othr_Tot_Clms"])[exact]
f["brand_hidden"] = f["Brnd_Sprsn_Flag"].notna().astype(int)
f["Brnd_Tot_Clms"] = brand.fillna(5).clip(lower=0).clip(upper=f["Tot_Clms"])

# --- Opioids ---
no_op = f["Opioid_Tot_Clms"] == 0
f["opioid_hidden"] = f["Opioid_Tot_Clms"].isna().astype(int)
f["Opioid_Tot_Clms"] = f["Opioid_Tot_Clms"].fillna(5)

f["opioid_benes_hidden"] = (f["Opioid_Tot_Benes"].isna() & ~no_op).astype(int)
f["Opioid_Tot_Benes"] = f["Opioid_Tot_Benes"].fillna(5).clip(upper=f["Opioid_Tot_Clms"])
f.loc[no_op, "Opioid_Tot_Benes"] = 0

f["opioid_la_hidden"] = (f["Opioid_LA_Tot_Clms"].isna() & ~no_op).astype(int)
f["Opioid_LA_Tot_Clms"] = f["Opioid_LA_Tot_Clms"].fillna(5).clip(upper=f["Opioid_Tot_Clms"])
f.loc[no_op, "Opioid_LA_Tot_Clms"] = 0
f.loc[no_op, "Opioid_Tot_Suply"] = 0          # no opioids = 0 days; otherwise leave blank

# --- Antibiotics, antipsychotics, patients ---
f["antibiotic_hidden"] = f["Antbtc_Tot_Clms"].isna().astype(int)
f["Antbtc_Tot_Clms"] = f["Antbtc_Tot_Clms"].fillna(5)

f["antipsych_hidden"] = f["Antpsyct_GE65_Sprsn_Flag"].eq("*").astype(int)
f["Antpsyct_GE65_Tot_Clms"] = f["Antpsyct_GE65_Tot_Clms"].fillna(5)

f["benes_hidden"] = f["Tot_Benes"].isna().astype(int)
f["Tot_Benes"] = f["Tot_Benes"].fillna(5)

# --- Check ---
hid = [c for c in f.columns if c.endswith("_hidden")]
print("Share of rows with each value estimated (%):")
print((f[hid].mean() * 100).round(1))
print("\nStill blank (on purpose):")
print(f[["Opioid_Tot_Suply", "GE65_Tot_Clms"]].isna().sum())
print("\nAny blanks left in filled columns?")
print(f[["Brnd_Tot_Clms", "Opioid_Tot_Clms", "Opioid_Tot_Benes", "Opioid_LA_Tot_Clms",
         "Antbtc_Tot_Clms", "Antpsyct_GE65_Tot_Clms", "Tot_Benes"]].isna().sum().sum())

In [0]:
f.attrs = {}
f.to_parquet("/Volumes/workspace/default/raw/partd_nc_filled.parquet", index=False)
print("Saved", len(f), "rows")

#### Raw features

In [0]:
x = f.copy()
safe = lambda a, b: a / b.where(b > 0)         # divide, but blank instead of dividing by 0

# A. How much
x["log_claims"]         = np.log1p(x["Tot_Clms"])
x["log_rx_per_patient"] = np.log(safe(x["Tot_Clms"], x["Tot_Benes"]))      # log: very skewed
x["log_cost"]           = np.log1p(x["Tot_Drug_Cst"])

# B. What kind
x["log_cost_per_rx"]    = np.log(safe(x["Tot_Drug_Cst"], x["Tot_Clms"]))   # log: very skewed
x["brand_share"]        = safe(x["Brnd_Tot_Clms"], x["Tot_Clms"])
x["days_per_rx"]        = safe(x["Tot_Day_Suply"], x["Tot_Clms"])

# C. Opioids and other drugs
x["opioid_rate"]          = safe(x["Opioid_Tot_Clms"], x["Tot_Clms"])
x["opioid_patient_share"] = safe(x["Opioid_Tot_Benes"], x["Tot_Benes"]).clip(upper=1)

x["la_share"] = safe(x["Opioid_LA_Tot_Clms"], x["Opioid_Tot_Clms"]).fillna(0)  # no opioids = 0
x.loc[x["opioid_hidden"] == 1, "la_share"] = np.nan                            # guess / guess = unknown

x["opioid_days_per_rx"] = safe(x["Opioid_Tot_Suply"], x["Opioid_Tot_Clms"])
x.loc[x["Opioid_Tot_Clms"] == 0, "opioid_days_per_rx"] = 0                     # no opioids = 0

x["antipsych_share"] = safe(x["Antpsyct_GE65_Tot_Clms"], x["GE65_Tot_Clms"]).clip(upper=1)
x.loc[x["GE65_Tot_Clms"] == 0, "antipsych_share"] = 0

x["antibiotic_rate"] = safe(x["Antbtc_Tot_Clms"], x["Tot_Clms"])

FEATURES = ["log_claims", "log_rx_per_patient", "log_cost", "log_cost_per_rx", "brand_share", "days_per_rx",
            "opioid_rate", "opioid_patient_share", "la_share", "opioid_days_per_rx",
            "antipsych_share", "antibiotic_rate"]
print(x[FEATURES].isna().sum())

In [0]:
x[FEATURES].describe(percentiles=[0.5, 0.99]).T.round(2)

In [0]:
# SET UP PEER GROUPS AND PATIENT-MIX COLUMNS
# - Peer group = the prescriber's specialty. Specialties with fewer than 30 prescribers
#   in a year are pooled into "Other" so every group is big enough to compare within.
# - rural = 1 if the prescriber is in a non-metro area (RUCA 4 to 10), 0 if metro.
#   A missing RUCA code counts as metro.
# - Any missing patient risk score or age is filled with the peer group's median.

a = x.copy()

n = a.groupby(["year", "Prscrbr_Type"])["Prscrbr_NPI"].transform("count")
a["peer"] = np.where(n >= 30, a["Prscrbr_Type"], "Other")

a["rural"] = (a["Prscrbr_RUCA"] >= 4).astype(int)

print("Missing RUCA:", a["Prscrbr_RUCA"].isna().sum(),
      "| missing risk:", a["Bene_Avg_Risk_Scre"].isna().sum(),
      "| missing age:", a["Bene_Avg_Age"].isna().sum())

for c in ["Bene_Avg_Risk_Scre", "Bene_Avg_Age"]:
    a[c] = a[c].fillna(a.groupby(["year", "peer"])[c].transform("median"))

print("Peer groups per year:", a.groupby("year")["peer"].nunique().to_dict())
print("Rows in 'Other':", (a["peer"] == "Other").sum())

In [0]:
# ADJUST EACH FEATURE FOR PATIENT MIX, THEN COMPARE TO PEERS
# For every specialty in every year, and for each of the 12 features:
#   1. Fit a robust (Huber) regression: feature ~ patient risk score + patient age + rural.
#      Huber gives less weight to extreme points, so outliers can't pull the line toward themselves.
#   2. Residual = actual value minus what the regression expected for a prescriber with those patients.
#      This is the part of the feature that patient mix CAN'T explain.
#   3. Turn residuals into robust z-scores: (residual - median) / MAD.
#      Uses median and MAD instead of mean and standard deviation, so outliers don't distort it.
#      Clipped at +/-10 so one extreme value can't dominate.
#   4. Unknown values (hidden opioid days, long-acting share, antipsychotics) get z = 0, meaning "assume typical".
# Result: 12 new columns starting with "z_". A z of +4 means "4 typical distances above peers
# with similar patients". May take a minute or two to run.

from sklearn.linear_model import HuberRegressor

ADJ = ["Bene_Avg_Risk_Scre", "Bene_Avg_Age", "rural"]

def robust_z(s):
    # Robust z-score: (value - median) / scale
    # Scale = MAD, but never smaller than the mean absolute deviation (a standard fallback
    # for "zero-heavy" data, where most values are 0 and the MAD collapses toward 0).
    med = s.median()
    mad = (s - med).abs().median() * 1.4826
    meanad = (s - med).abs().mean() * 1.2533
    scale = max(mad, meanad)
    if not scale or np.isnan(scale):
        return s * 0
    return ((s - med) / scale).clip(-10, 10)

def adjust(g):
    out = pd.DataFrame(index=g.index)
    X = g[ADJ].to_numpy(float)
    sd = X.std(0); sd[sd == 0] = 1
    X = (X - X.mean(0)) / sd                         # standardize adjusters so Huber behaves
    for feat in FEATURES:
        y = g[feat]
        ok = y.notna().to_numpy()
        resid = pd.Series(np.nan, index=g.index)
        if ok.sum() >= 30 and y[ok].std() > 0:
            try:
                m = HuberRegressor(max_iter=500).fit(X[ok], y[ok])
                resid[ok] = y[ok] - m.predict(X[ok])
            except Exception:
                resid[ok] = y[ok] - y[ok].median()
        else:
            resid[ok] = y[ok] - y[ok].median()
        out["z_" + feat] = robust_z(resid)
    return out

z = a.groupby(["year", "peer"], group_keys=False).apply(adjust, include_groups=False)
a = a.join(z)
ZF = ["z_" + f for f in FEATURES]

print("Unknowns before filling (get the typical value, z = 0):")
print(a[ZF].isna().sum()[lambda s: s > 0])
a[ZF] = a[ZF].fillna(0)

In [0]:
# DID THE ADJUSTMENT WORK?
# Correlation between each adjusted feature (z_...) and the patient-mix columns.
# If the adjustment worked, every number should be close to 0 (roughly -0.05 to 0.05),
# meaning patient risk, age, and rural vs urban no longer drive the features.

check = a[ZF + ADJ].corr().loc[ZF, ADJ].round(3)
check

In [0]:
#  BEFORE vs AFTER ADJUSTMENT
# Same check, but on the raw features, to show how much patient mix mattered before adjusting.
before = a[FEATURES + ADJ].corr().loc[FEATURES, ADJ].round(3)
before

### One row per prescriber

In [0]:
# COMBINE EACH PRESCRIBER'S YEARS INTO ONE ROW
# - The 12 adjusted features (z_...) are averaged across the prescriber's years.
# - Two "change over time" features compare their LAST year to their FIRST year:
#     opioid_trend  = z_opioid_rate (last) - z_opioid_rate (first)
#                     Positive = they moved UP relative to peers while most cut back ("against the trend")
#     volume_growth = z_log_claims (last) - z_log_claims (first)
#                     Positive = their prescribing volume jumped relative to peers
#   Because these use z-scores, they measure change compared to peers in the same year,
#   not raw change (which would be misleading since everyone's opioid rate dropped).
# - Also keeps info for later (not model features): specialty, location, patient mix,
#   low-income share, and how often values were estimated.

a = a.sort_values(["Prscrbr_NPI", "year"])
a["lis_share"] = a["LIS_Tot_Clms"] / a["Tot_Clms"]
g = a.groupby("Prscrbr_NPI")

feat = g[ZF].mean()
first = g[["z_opioid_rate", "z_log_claims"]].first()
last = g[["z_opioid_rate", "z_log_claims"]].last()
feat["opioid_trend"] = last["z_opioid_rate"] - first["z_opioid_rate"]
feat["volume_growth"] = last["z_log_claims"] - first["z_log_claims"]

info = g.agg(
    last_name=("Prscrbr_Last_Org_Name", "last"), first_name=("Prscrbr_First_Name", "last"),
    state=("Prscrbr_State_Abrvtn", "last"), city=("Prscrbr_City", "last"), zip5=("Prscrbr_Zip5", "last"),
    specialty=("Prscrbr_Type", "last"), peer=("peer", "last"), rural=("rural", "last"),
    n_years=("year", "nunique"), first_year=("year", "min"), last_year=("year", "max"),
    avg_claims=("Tot_Clms", "mean"), avg_risk=("Bene_Avg_Risk_Scre", "mean"),
    lis_share=("lis_share", "mean"),
    opioid_hidden=("opioid_hidden", "mean"), brand_hidden=("brand_hidden", "mean"),
)

prescribers = info.join(feat).reset_index()
MODEL_FEATURES = ZF + ["opioid_trend", "volume_growth"]

print(f"{len(prescribers):,} prescribers, {len(MODEL_FEATURES)} model features")
prescribers[MODEL_FEATURES].describe(percentiles=[0.5, 0.99]).T.round(2)

In [0]:
# SAVE THE FINAL FEATURE TABLE
# One row per prescriber, ready for the model in Step 3.
# Names and location stay in Databricks only (needed to match the OIG list later);
# they are never published to GitHub or the dashboard.

prescribers.attrs = {}
prescribers.to_parquet("/Volumes/workspace/default/raw/prescriber_features.parquet", index=False)
print("Saved", len(prescribers), "prescribers")

In [0]:
#  HOW MANY ROWS HIT THE +/-10 CAP?
(a[ZF].abs() >= 10).mean().mul(100).round(1).sort_values(ascending=False)